Complete regularization setup in PyTorch

Combining dropout, weight decay (AdamW), and early stopping in a training loop




In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# model with dropout
class RegularizedNet(nn.Module):
  def __init__(self, dropout=0.3):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(784,256),
        nn.ReLU(),
        nn.Dropout(p=dropout), # dropout after activation
        nn.Linear(256,128),
        nn.ReLU(),
        nn.Dropout(p=dropout),
        nn.Linear(128,10)
    )
  def forward(self, x):
    return self.layers(x)

model = RegularizedNet(dropout=0.3)

# ============================================================
# L2 WEIGHT DECAY (use AdamW, not Adam)
# AdamW decouples weight decay from gradient updates
# for more uniform regularization
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(),
                              lr = 1e-3,
                              weight_decay = 0.01) #l2 penalty strength
# EARLY STOPPING

class EarlyStopping:
  def __init__(self, patience = 5, min_delta = 0.001):
    self.patience = patience
    self.min_delta = min_delta
    self.counter = 0
    self.best_loss = float('inf')
    self.best_model = None

  def __call__(self, val_loss, model):
    if val_loss < self.best_loss - self.min_delta:
      self.best_loss = val_loss
      self.best_model = {
          k: v.clone() for k,v in model.state_dict().items()
      }
      self.counter = 0
    else:
      self.counter += 1
    return self.counter >= self.patience

  def restore_best(self, model):
    model.load_state_dict(self.best_model)

# training loop with all three techniques
early_stop = EarlyStopping(patience=7, min_delta = 0.001)
criterion = nn.CrossEntropyLoss()

# synthetic data for demonstration
train_data = TensorDataset(torch.randn(320,784), torch.randint(0, 10, (320,)))
val_data = TensorDataset(torch.randn(64,784), torch.randint(0, 10, (64,)))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

for epoch in range(100):
  model.train()
  train_loss = 0
  for x, y in train_loader:
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss /= len(train_loader)

 # validation dropout disabled
  model.eval()
  val_loss = 0
  with torch.no_grad():
    for x,y in val_loader:
      val_loss += criterion(model(x), y).item()

  val_loss /= len(val_loader) #average over batches
  print(f"Epoch {epoch}: train={train_loss:.3f}, val={val_loss:.3f}")

  if(early_stop(val_loss, model)):
    print(f"Early Stopping at epoch {epoch}")
    early_stop.restore_best(model)
    break


Epoch 0: train=2.324, val=2.269
Epoch 1: train=2.100, val=2.261
Epoch 2: train=1.866, val=2.257
Epoch 3: train=1.481, val=2.274
Epoch 4: train=0.975, val=2.301
Epoch 5: train=0.474, val=2.409
Epoch 6: train=0.233, val=2.555
Epoch 7: train=0.079, val=2.639
Epoch 8: train=0.046, val=2.708
Epoch 9: train=0.033, val=2.774
Early Stopping at epoch 9


Dropout rates by architecture

Recommended dropout values for CNNs, Transformers, and RNNs

In [10]:
import torch.nn as nn

# ============================================================
# CNN: No dropout in conv layers, high in FC layers
# Use data augmentation as primary regularization
# ============================================================

class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 64, padding=1, bias=False), # bias = false because batchnorm has its own bias
        nn.BatchNorm(64), # batch norm provides implicit regularization
        nn.ReLU(),
        nn.MaxPool2d(2),
        # no dropout in conv layers(or very low 0.1)
    )
    self.classifier = nn.Sequential(
        nn.Dropout(0.5), # High Dropout before FC Layers
        nn.Linear(64 * 16 * 16 * 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256,10)
    )
# ============================================================
# TRANSFORMER: Moderate dropout in attention and FFN
# ============================================================

class TransformersBlock(nn.Module):
  def __init__(self, d_model=512, dropout=0.1):
    super().__init__()
    self.attn = nn.MultiheadAttention(d_model, 8, dropout=dropout)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(d_model * 4, d_model)
    )
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout) # residual dropout

  def forward(self, x):
    # Pre-LN: normalize before sublayer (more stable training)
    attn_out = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
    x = x + self.dropout(attn_out)
    x = x + self.dropout(self.ffn(self.norm2(x)))
    return x

# ============================================================
# QUICK REFERENCE TABLE
# ============================================================
# Architecture    | Conv/Attn    | FC/FFN     | Weight Decay
# ----------------+--------------+------------+-------------
# CNN             | 0.0 - 0.2   | 0.5        | 0.0001 (SGD)
# Transformer     | 0.1          | 0.1        | 0.01-0.1 (AdamW)
# RNN/LSTM        | 0.2 - 0.5   | 0.3 - 0.5  | 0.0001
# MLP             | N/A          | 0.3 - 0.5  | 0.01
# Modern LLMs     | 0.0          | 0.0        | 0.1 (AdamW only)

### Transformer Block Demonstration

Here's an example of how to use the `TransformersBlock` with some synthetic data. We'll initialize the block and then pass a random tensor through it to observe the output.

In [11]:
import torch

# Define the dimensions for the synthetic data
batch_size = 4
sequence_length = 10
d_model = 512 # Must match d_model in TransformersBlock

# Create an instance of the TransformersBlock
transformer_block = TransformersBlock(d_model=d_model, dropout=0.1)

# Generate synthetic input data
# Input should be of shape (sequence_length, batch_size, d_model) for MultiheadAttention
# Or, if batch_first=True is used in MultiheadAttention, then (batch_size, sequence_length, d_model)
# Assuming default (sequence_length, batch_size, d_model) for this example
synthetic_input = torch.randn(sequence_length, batch_size, d_model)

print(f"Input shape: {synthetic_input.shape}")

# Pass the synthetic data through the transformer block
# Set the model to evaluation mode if dropout should be inactive
transformer_block.eval() # Or .train() if you want dropout active
with torch.no_grad(): # No need to calculate gradients for a simple forward pass demo
    output = transformer_block(synthetic_input)

print(f"Output shape: {output.shape}")

Input shape: torch.Size([10, 4, 512])
Output shape: torch.Size([10, 4, 512])
